# RadiologyAI — Linha de base honesta no Google Colab

**O que este notebook produz:** *um* número de desempenho real, medido, reproduzível,
com intervalo de confiança — e o artefato `metrics.json` que o sustenta.

É o marco da Fase 1 do [ROADMAP](https://github.com/drguilhermecapel/radiologyai/blob/main/ROADMAP.md):
substituir todas as métricas fabricadas do repositório v1 por uma medição verificável.

---

## A armadilha que este notebook evita

O caminho óbvio seria rodar `densenet121-res224-all` no NIH ChestX-ray14. **Não faça isso.**

Os pesos `-all` foram treinados em NIH, PadChest, CheXpert, MIMIC-CXR, Google, OpenI e RSNA —
o próprio nome do arquivo publicado é `nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-...`.
Avaliá-los no NIH é **in-distribution**: produziria um número inflado que parece medição mas não é.
Seria uma nova fabricação, só que mais sutil que a do v1.

Este notebook usa **`densenet121-res224-pc`** — treinado só em PadChest (Hospital San Juan,
Alicante, Espanha) — avaliado no **split oficial de teste do NIH** (EUA, 25.596 imagens,
disjunto por paciente). País, equipamento, população e pipeline de rotulagem diferentes:
**validação externa genuína, sem credenciamento, custo zero.**

O código detecta vazamento automaticamente e recusa alegar validação externa quando não há.

---

## Antes de começar

1. `Ambiente de execução -> Alterar o tipo de ambiente de execução -> GPU T4`
2. A GPU não é obrigatória (funciona em CPU), mas reduz a inferência de ~2h para ~12min.
3. O disco do Colab é **efêmero**. O dataset (42 GB) é baixado a cada sessão; só os
   artefatos (poucos KB) são salvos no seu Drive.


## 1. Ambiente


In [ ]:
!nvidia-smi 2>/dev/null || echo 'Sem GPU — vai funcionar em CPU, só mais devagar'
!df -h /content | tail -1
import sys; print('Python', sys.version.split()[0])


O Colab roda Python 3.12; o pacote exige 3.11. Instalamos sem a checagem de versão
(`--no-deps` no próprio pacote) e as dependências explicitamente. É uma concessão
consciente ao ambiente do Colab: o pino `>=3.11,<3.12` continua valendo para CI e produção.


In [ ]:
REPO = 'https://github.com/drguilhermecapel/radiologyai.git'
BRANCH = 'claude/roadmap-interpretacao-radiologica-vh15ek'

!git clone --depth 1 --branch {BRANCH} {REPO} /content/radiologyai
%cd /content/radiologyai
!git log --oneline -1


In [ ]:
# Dependências. torch já vem no Colab.
!pip install -q pydicom==2.4.4 'pydantic>=2.7' 'pydantic-settings>=2.3' \
    'typer>=0.12' torchxrayvision scikit-image pillow
!pip install -q --no-deps -e .

import sys; sys.path.insert(0, '/content/radiologyai/src')
import radiologyai; print('radiologyai', radiologyai.__version__)


## 2. Montar o Drive (só para os artefatos)

O dataset **não** vai para o Drive — 42 GB não cabem na conta gratuita de 15 GB, e não
precisam: os pixels são reproduzíveis a partir da fonte. O que persiste é o `metrics.json`
e o manifest, que somam poucos megabytes e são o registro de reprodutibilidade.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
ARTIFACTS = Path('/content/drive/MyDrive/radiologyai/artifacts/eval')
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print('artefatos serão salvos em:', ARTIFACTS)


## 3. Baixar o NIH ChestX-ray14

112.120 radiografias frontais de 30.805 pacientes, do NIH Clinical Center.
Livre, sem registro e sem credenciamento.

**Duas rotas.** A do Kaggle é bem mais rápida e resumível; a do NIH Box não exige conta.
Escolha uma abaixo.


In [ ]:
DATA = Path('/content/nih')
DATA.mkdir(parents=True, exist_ok=True)

# ROTA A — Kaggle (recomendada: mais rápida, resumível)
# Pegue kaggle.json em https://www.kaggle.com/settings -> Create New Token
USE_KAGGLE = True

if USE_KAGGLE:
    from google.colab import files
    import os, json
    if not os.path.exists('/root/.kaggle/kaggle.json'):
        print('Envie seu kaggle.json:')
        files.upload()
        os.makedirs('/root/.kaggle', exist_ok=True)
        os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
        os.chmod('/root/.kaggle/kaggle.json', 0o600)
    !pip install -q kaggle
    !kaggle datasets download -d nih-chest-xrays/data -p {DATA} --unzip
else:
    # ROTA B — links oficiais do NIH (Box). Sem conta, porém mais lento.
    !python scripts/fetch_nih_cxr14.py --out {DATA}


In [ ]:
# Os três arquivos oficiais precisam existir. O split de teste do NIH já é
# disjunto por paciente — é por isso que usamos o oficial em vez de fazer o nosso.
for name in ['Data_Entry_2017_v2020.csv', 'test_list.txt']:
    p = DATA / name
    print(('OK  ' if p.exists() else 'FALTA '), name)

!ls {DATA} | head
!du -sh {DATA}


## 4. Construir o manifest

O manifest é o registro de reprodutibilidade: quais imagens, de qual paciente, com quais
rótulos. Ele é commitado no repositório; os pixels nunca são.


In [ ]:
!python -m radiologyai.cli.main manifest {DATA} \
    --out /content/radiologyai/datasets/manifests/nih_cxr14_test.csv \
    --split test


## 5. Rodar a avaliação

Este é o único caminho pelo qual um número de desempenho pode entrar no repositório.
O artefato grava `git_sha`, `weights_sha256`, `manifest_sha256`, `seed` e as versões
das bibliotecas — o conjunto necessário para demonstrar que a execução é repetível
(IEC 62304 §5.7).

Tempo estimado: ~12 min em T4, ~2 h em CPU.


In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('dispositivo:', DEVICE)

!python -m radiologyai.cli.main evaluate \
    --card xrv-densenet121-pc \
    --manifest /content/radiologyai/datasets/manifests/nih_cxr14_test.csv \
    --data-root {DATA} \
    --out {ARTIFACTS} \
    --device {DEVICE} \
    --batch-size 64 \
    --bootstrap 2000 \
    --seed 20260101


## 6. Ler o resultado

**Expectativa: AUROC macro entre 0,72 e 0,82.** Cardiomegalia, derrame e enfisema devem
ficar altos (0,85–0,90); pneumonia, infiltrado e nódulo baixos (0,65–0,73).

O README do v1 alegava 0,94. **Publicar 0,78 com intervalo de confiança é o objetivo
inteiro deste marco.** Um número menor e verdadeiro vale mais que um número maior e inventado.


In [ ]:
import json
run_dir = sorted(ARTIFACTS.glob('xrv-densenet121-pc__*'))[-1]
m = json.loads((run_dir / 'metrics.json').read_text())

print('AUROC macro:', m['macro_auroc'])
print('rótulos avaliados:', m['n_labels_evaluated'])
print('externo ao treino:', m['dataset']['external_to_training_data'],
      f"({m['dataset']['leakage_status']})")
print('imagens/pacientes:', m['dataset']['n_images'], '/', m['dataset']['n_patients'])
print()
for name, e in sorted(m['per_label'].items(), key=lambda kv: -kv[1]['auroc']):
    lo, hi = e['auroc_ci95']
    print(f"  {name:<26} {e['auroc']:.3f}  IC95 [{lo:.3f}, {hi:.3f}]  n+={e['n_pos']:>5}")


In [ ]:
# Subgrupos — requisito de equidade. Uma lacuna grande é achado a reportar,
# não a esconder. A diferença PA vs AP costuma ser visível e é ela própria um achado.
for group, values in m['subgroups'].items():
    print(f'\n{group}:')
    for k, v in values.items():
        print(f"  {k:<12} n={v['n']:>6}  AUROC macro={v['macro_auroc']}")


In [ ]:
# O que NÃO foi avaliado — declarado, nunca descartado em silêncio
for x in m['not_evaluated']:
    print(f"  {x['label']:<28} {x['reason']}")
print()
print('LIMITAÇÕES DECLARADAS:')
for l in m['limitations']:
    print(' *', l)


## 7. Levar o resultado de volta ao repositório

O artefato é o que autoriza qualquer alegação. Sem ele, o verificador de CI
`scripts/check_honesty.py` quebra o build de qualquer documento que cite um número.


In [ ]:
import shutil
dest = Path('/content/radiologyai/artifacts/eval') / run_dir.name
dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(run_dir, dest, dirs_exist_ok=True)

!cd /content/radiologyai && python scripts/check_honesty.py
print('\nArquivos a commitar:')
!cd /content/radiologyai && git status --short artifacts/ datasets/


In [ ]:
# Baixe para o seu computador e commite no repositório
from google.colab import files
shutil.make_archive('/content/baseline', 'zip', run_dir)
files.download('/content/baseline.zip')


---

## O que este resultado é — e o que não é

**É:** uma medição retrospectiva de desempenho de algoritmo isolado, num conjunto de
teste externo ao treino do modelo, com intervalo de confiança e análise de subgrupos.

**Não é:** validação clínica. Não é evidência de utilidade clínica. Não é medição do
modelo *do produto* — este é um modelo de terceiros usado como referência. Os escores
não são calibrados e não representam probabilidade de doença.

Os rótulos do NIH são **minerados por NLP dos laudos**, não adjudicados por radiologista.
A validação contra rótulos adjudicados vem na Fase 3, com o **VinDr-CXR** (18.000 exames,
conjunto de teste lido por 5 radiologistas) — que exige credenciamento PhysioNet.

**Comece o credenciamento agora:** curso CITI "Data or Specimens Only Research", gratuito,
~6 h, e a aprovação leva de 2 a 6 semanas. Como médico com CRM ativo você é elegível.
Este notebook foi desenhado para não depender de nada credenciado, justamente para que
o credenciamento corra em paralelo e nunca fique no caminho crítico.
